# Lecture 23: The Canonical Linear Model — Geometric Foundations

**Data 145, Spring 2026: Evidence and Uncertainty**
**Instructors:** Ani Adhikari, William Fithian

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# --- Plot defaults ---
plt.rcParams.update({
    'figure.figsize': (10, 4),
    'font.size': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## Introduction

This week we pivot from hypothesis testing in one-parameter problems to a much richer setting: testing linear hypotheses about the mean of a multivariate normal. The tools we'll build — the **canonical linear model** and its geometric interpretation — underlie a huge swath of classical statistics: the one-sample $t$-test, the two-sample $t$-test, analysis of variance, and linear regression. All of them turn out to be the *same problem in different coordinate systems.*

The key idea is geometric. If $Z \sim N_n(\mu, \sigma^2 I_n)$, then $Z$ lives in $\mathbb R^n$ and we can decompose it into orthogonal pieces. Each piece is independent of the others (a property we inherit from the multivariate normal), and we can form test statistics by comparing the magnitude of $Z$'s projection onto the "signal direction" to its projection onto a "residual direction." Depending on whether $\sigma^2$ is known and whether the signal direction is 1D or higher-dimensional, we get $z$-, $\chi^2$-, $t$-, or $F$-tests — all variations on one theme.

Today we lay the foundations:

1. **Probability review.** The $\chi^2$, $t$, and $F$ distributions, recast as quantities built from iid standard normals, and the key fact that $N_n(0, \sigma^2 I_n)$ is rotationally invariant.
2. **Canonical linear model, $n = 2$.** The simplest version of the setup we'll generalize: $Z \sim N_2(\binom{\mu_1}{0}, \sigma^2 I_2)$, test $H_0: \mu_1 = 0$.
3. **Rotation into canonical form.** The familiar one-sample $t$-test with $n=2$ is literally this problem in disguise — a rotation of the coordinate axes reveals the structure.
4. **General-$n$ one-sample $t$-test.** Same geometry, extended to $n$ observations by introducing notation for the signal direction ($Z_1$) and the residual directions ($Z_r$).
5. **Preview of Thursday.** An $n = 3$ warm-up with a *nuisance* direction $Z_0$, setting up the general canonical linear model we'll develop next lecture.

A data-review reminder: you've seen multivariate normals, covariance matrices, and their behavior under linear transformations in Data 140 (Chapter 23). If any of that feels rusty, skim those notes before class.

---

## 1. Probability Review: Gaussian-Adjacent Distributions

The sampling distributions we'll meet this week — $\chi^2$, $t$, $F$ — all arise from arithmetic on iid standard normals. Let's collect the definitions in one place and highlight the unifying theme.

### 1.1 Chi-Squared

**Definition.** If $Z_1, \ldots, Z_d \stackrel{iid}{\sim} N(0, 1)$, then
$$V = \sum_{i=1}^d Z_i^2 \sim \chi^2_d.$$

From Data 140 (stated without proof): $\chi^2_d = \text{Gamma}(d/2,\ 2)$, so $E[V] = d$ and $\text{Var}(V) = 2d$.

As a consequence of the CLT, $(V - d)/\sqrt{2d} \to N(0, 1)$ as $d \to \infty$.

### 1.2 $t$-Distribution

**Definition.** If $Z \sim N(0, 1)$ and $V \sim \chi^2_d$ are independent, then
$$T = \frac{Z}{\sqrt{V/d}} \sim t_d.$$

Why divide $V$ by $d$? So that $V/d$ has mean 1 — the denominator is an unbiased estimate of 1, which is the variance of $Z$. This is the template we'll use again and again: standardize a Gaussian numerator by dividing by the square root of an unbiased estimate of its variance.

**Why this is so useful: $\sigma$ cancels.** If we instead had $\tilde Z = \sigma Z \sim N(0, \sigma^2)$ and $\tilde V = \sigma^2 V \sim \sigma^2 \chi^2_d$ (independent), then
$$\frac{\tilde Z}{\sqrt{\tilde V/d}} = \frac{\sigma Z}{\sqrt{\sigma^2 V/d}} = \frac{Z}{\sqrt{V/d}} \sim t_d.$$
The unknown $\sigma$ cancels, so the ratio is a pivotal quantity — we can compute it from data without knowing $\sigma$, and its distribution is known. This is exactly why the $t$-distribution is the right reference for inference with unknown $\sigma^2$.

As $d \to \infty$, $V/d \to 1$ by the law of large numbers, so $T \to Z \sim N(0, 1)$. For small $d$ the $t$-distribution has heavier tails than $N(0, 1)$; this reflects our uncertainty about $\sigma^2$.

You met this distribution in Lecture 13 as the null distribution of the one-sample $t$-statistic $\sqrt n (\bar X - \mu_0)/S$. Today we'll see *geometrically* why that statistic has this form.

### 1.3 $F$-Distribution

**Definition.** If $V_1 \sim \chi^2_{d_1}$ and $V_2 \sim \chi^2_{d_2}$ are independent, then
$$F = \frac{V_1/d_1}{V_2/d_2} \sim F_{d_1, d_2}.$$

Both $V_1/d_1$ and $V_2/d_2$ are unbiased estimates of 1 (if scaled to represent equal variances), so $F$ is the ratio of two such estimates.

You derived the density of $F_{d_1, d_2}$ on Worksheet 9 using the beta–gamma algebra from Lecture 22. We won't re-derive it here.

**A key identity.** If $T \sim t_d$, then $T^2 \sim F_{1, d}$. This is immediate from the definitions:
$$T^2 = \frac{Z^2}{V/d} = \frac{Z^2/1}{V/d},$$
where $Z^2 \sim \chi^2_1$ and is independent of $V$. So the two-sided $t$-test is exactly the $F$-test with numerator degrees of freedom $d_1 = 1$.

### 1.4 The Key New Hook: Rotational Invariance of $N_n(0, \sigma^2 I_n)$

Recall from Data 140 that the multivariate normal is closed under affine transformations: if $Z \sim N_n(\mu, \Sigma)$ and $Y = AZ + b$ for a matrix $A$ and vector $b$, then
$$Y \sim N(A\mu + b,\ A \Sigma A^T).$$

We'll call this the **affine transformation formula**.

**Apply it with $\mu = 0$, $\Sigma = \sigma^2 I_n$, $b = 0$, and $A = Q$ an orthogonal matrix** ($Q^T Q = I_n$). Then:
$$Q Z \sim N_n(0,\ \sigma^2 Q Q^T) = N_n(0,\ \sigma^2 I_n).$$

So $QZ \stackrel{d}{=} Z$: the distribution of $Z$ is **rotationally invariant**. Rotating or reflecting $Z$ produces a vector with the same distribution as $Z$.

**Consequence we'll use all day.** Let $V_1, V_2$ be orthogonal subspaces of $\mathbb R^n$ with $\dim V_1 = d_1$, $\dim V_2 = d_2$, and suppose $Z \sim N_n(0, \sigma^2 I_n)$. Write $P_{V_1} Z$ for the orthogonal projection of $Z$ onto $V_1$. Then:

- $\|P_{V_1} Z\|^2 \sim \sigma^2 \chi^2_{d_1}$ and $\|P_{V_2} Z\|^2 \sim \sigma^2 \chi^2_{d_2}$.
- These are **independent**.

*Why?* Stack orthonormal bases for $V_1$ and $V_2$ (and for any remaining orthogonal complement) into an orthogonal matrix $Q$. Rotate: $Z' = Q^T Z \sim N_n(0, \sigma^2 I_n)$, so $Z'$ has iid $N(0, \sigma^2)$ coordinates. The projection magnitudes are sums of squared coordinates from disjoint blocks, hence independent, and each block is $\sigma^2 \chi^2$ with degrees of freedom equal to the block size.

This is the engine behind every test we'll build today: **a test statistic is a ratio of two projection magnitudes onto orthogonal subspaces.**

---

## 2. The Canonical Linear Model: $n = 2$ Case

### 2.1 Setup

Suppose we observe
$$Z \sim N_2\!\left(\binom{\mu_1}{0},\ \sigma^2 I_2\right),$$
and we want to test
$$H_0: \mu_1 = 0 \quad \text{vs} \quad H_1: \mu_1 \neq 0.$$

Two observations, $Z_1$ and $Z_2$. Only $Z_1$ carries any signal — its mean equals $\mu_1$, which is $0$ under the null and nonzero under the alternative. The second observation $Z_2$ has mean $0$ under *both* hypotheses; it's **pure noise**. Because the covariance matrix is diagonal and the vector is MVN, $Z_1 \perp\!\!\!\perp Z_2$.

This simple setup is the prototype of everything else we'll do. Let's work out the tests.

### 2.2 $\sigma^2$ Known: $z$-Test and $\chi^2_1$-Test

When $\sigma^2$ is known, the natural test statistic is
$$\frac{Z_1}{\sigma} \sim N(0, 1) \quad \text{under } H_0.$$
Reject if $|Z_1/\sigma| > z_{\alpha/2}$. This is the two-sided $z$-test.

We can equivalently square the statistic:
$$\left(\frac{Z_1}{\sigma}\right)^2 \sim \chi^2_1 \quad \text{under } H_0.$$
Reject if this exceeds $\chi^2_1(\alpha)$ — the upper-$\alpha$ quantile of the $\chi^2_1$ distribution. This is the $\chi^2_1$-test.

The two-sided $z$-test and the $\chi^2_1$-test are **exactly the same test**: the $\chi^2_1$ statistic is the squared $z$, and the rejection regions coincide. Why state the squared form separately? Because it generalizes cleanly when the signal is higher-dimensional (e.g., $\chi^2_{d_1}$ for $d_1$-dimensional signals) — a preview of what we'll develop on Thursday.

**One-sided alternatives.** The signed statistic $Z_1/\sigma$ retains the sign of $Z_1$, so we can test $H_0: \mu_1 \leq 0$ vs $H_1: \mu_1 > 0$ by rejecting when $Z_1/\sigma > z_\alpha$. The $\chi^2_1$ form, being a squared quantity, is inherently two-sided.

### 2.3 $\sigma^2$ Unknown: $t_1$-Test and $F_{1,1}$-Test

Now suppose we don't know $\sigma^2$. We can't compute $Z_1/\sigma$. What do we do?

**Key observation.** Under both $H_0$ and $H_1$, $Z_2 \sim N(0, \sigma^2)$. So $Z_2^2$ is an unbiased estimate of $\sigma^2$ — regardless of $\mu_1$. The residual coordinate gives us a variance estimate for free.

**Construction.** Form the $t$-statistic by standardizing $Z_1$ using $Z_2$ in place of $\sigma$:
$$T = \frac{Z_1}{\sqrt{Z_2^2}} = \frac{Z_1 / \sigma}{\sqrt{Z_2^2 / \sigma^2}} = \frac{N(0, 1)}{\sqrt{\chi^2_1 / 1}} \sim t_1 \quad \text{under } H_0.$$

The numerator is standard normal, the denominator is the square root of an independent $\chi^2_1$ divided by its degrees of freedom. By definition this ratio is $t_1$-distributed. Reject if $|T| > t_1(\alpha/2)$.

**Aside: $t_1$ is the Cauchy distribution.** It has no mean or variance — heavy tails. The reason is visible in the construction: the denominator $|Z_2|$ can get arbitrarily close to zero, blowing up the ratio. This effect softens at larger degrees of freedom because the density of a $\chi^2_d$ variable near zero goes rapidly to zero as $d$ grows.

**Squared form.** Squaring: $T^2 = Z_1^2 / Z_2^2 \sim F_{1,1}$. Reject if $T^2 > F_{1,1}(\alpha)$. Again the two-sided $t_1$-test and the $F_{1,1}$-test are the same test.

### 2.4 Geometric Picture

Plot the observation $Z$ in the plane with axes $Z_1$ (horizontal) and $Z_2$ (vertical). The **signal direction** is the $Z_1$-axis and the **residual direction** is the $Z_2$-axis.

Under $H_0$: $Z$ is centered at the origin and rotationally symmetric (contours are circles).
Under $H_1$: $Z$ is centered at $(\mu_1, 0)$ — shifted along the signal axis.

**Reading off the test statistic geometrically.** Suppose $\theta$ is the angle between the observed $Z$ and the $Z_1$-axis. Then
$$\cos\theta = \frac{Z_1}{\|Z\|}, \qquad \sin\theta = \frac{Z_2}{\|Z\|},$$
so
$$\frac{Z_1}{Z_2} = \cot\theta.$$

The $t_1$-statistic is (up to sign) $|Z_1|/|Z_2| = |\cot\theta|$: it measures **how aligned $Z$ is with the signal direction**. When $Z$ hugs the $Z_1$-axis (small $\theta$), the statistic is large; when $Z$ hugs the $Z_2$-axis (large $\theta$), it's small. Crucially, this reading works **without knowing $\sigma^2$**, because $\sigma^2$ simply sets the scale of the picture — not the angle. A $Z$-vector tightly hugging the $Z_1$-axis is surprising under the rotationally-symmetric null no matter what $\sigma^2$ is.

---

## 3. The One-Sample $t$-Test ($n = 2$) Is Canonical in Disguise

### 3.1 Setup

Now consider the familiar one-sample $t$-test with $n = 2$:
$$X_1, X_2 \stackrel{iid}{\sim} N(\mu, \sigma^2), \qquad H_0: \mu = 0 \quad \text{vs} \quad H_1: \mu \neq 0.$$

Here $X = (X_1, X_2)^T$ has mean vector $\mu \mathbf 1 = (\mu, \mu)^T$ — both coordinates share the same mean. This is *not* the form we analyzed in §2, where only $Z_1$ had a nonzero mean. But a well-chosen rotation will convert it to that form.

### 3.2 The Rotation

Let
$$Q = \frac{1}{\sqrt 2} \begin{pmatrix} 1 & -1 \\ 1 & 1 \end{pmatrix}.$$

This is a proper $45°$ rotation matrix ($\det Q = 1$). Its columns are orthonormal:

- First column: $\mathbf 1/\sqrt 2 = (1/\sqrt 2,\ 1/\sqrt 2)^T$ — the unit vector along the **signal direction** $(1, 1)$.
- Second column: $(-1/\sqrt 2,\ 1/\sqrt 2)^T$ — the unit vector $90°$ counterclockwise, along the **residual direction**.

Check: each column has length 1, and they're orthogonal. So $Q$ is orthogonal ($Q^T Q = I_2$).

Define the **rotated coordinates**
$$Z = Q^T X = \frac{1}{\sqrt 2} \begin{pmatrix} 1 & 1 \\ -1 & 1 \end{pmatrix} \begin{pmatrix} X_1 \\ X_2 \end{pmatrix} = \begin{pmatrix} (X_1 + X_2)/\sqrt 2 \\ (X_2 - X_1)/\sqrt 2 \end{pmatrix}.$$

By the affine transformation formula,
$$Z \sim N_2(Q^T \mu \mathbf 1,\ \sigma^2 Q^T Q) = N_2\!\left(\binom{\sqrt 2\, \mu}{0},\ \sigma^2 I_2\right).$$

Reading off:

- $Z_1 = (X_1 + X_2)/\sqrt 2 \sim N(\sqrt 2\, \mu,\ \sigma^2)$.
- $Z_2 = (X_2 - X_1)/\sqrt 2 \sim N(0,\ \sigma^2)$.
- $Z_1 \perp\!\!\!\perp Z_2$.

**Reparametrize** by $\mu_1 = \sqrt 2\, \mu$. Then $H_0: \mu = 0 \iff H_0: \mu_1 = 0$. The rotated problem is **exactly** the §2 setup: $Z \sim N_2(\binom{\mu_1}{0}, \sigma^2 I_2)$, test $\mu_1 = 0$.

### 3.3 Picture

In the $X_1, X_2$ plane:

- The line $X_1 = X_2$ is the **signal direction**: the mean vector $\mu \mathbf 1 = (\mu, \mu)^T$ lives on this line.
- The line $X_1 = -X_2$ is the **residual direction**: orthogonal to the signal.

Rotating the axes by $45°$ counterclockwise aligns the signal direction with the horizontal — that's the canonical $(Z_1, Z_2)$ picture from §2. Same data, new basis.

### 3.4 Back-Translation

In the rotated coordinates, the $t_1$-test statistic from §2 is
$$T = \frac{Z_1}{|Z_2|} = \frac{(X_1 + X_2)/\sqrt 2}{|X_2 - X_1|/\sqrt 2} = \frac{X_1 + X_2}{|X_1 - X_2|}$$
(using $|X_2 - X_1| = |X_1 - X_2|$).

Let's verify this matches the **classical one-sample $t$-statistic** $T_{\text{classical}} = \sqrt 2\, \bar X / S$.

Compute $S^2$ for $n = 2$. The sample mean is $\bar X = (X_1 + X_2)/2$, so
$$X_1 - \bar X = X_1 - \frac{X_1 + X_2}{2} = \frac{X_1 - X_2}{2}, \qquad X_2 - \bar X = -\frac{X_1 - X_2}{2}.$$

Therefore
$$S^2 = \sum_{i=1}^2 (X_i - \bar X)^2 = 2 \cdot \left(\frac{X_1 - X_2}{2}\right)^2 = \frac{(X_1 - X_2)^2}{2},$$
(using $n - 1 = 1$), so $S = |X_1 - X_2|/\sqrt 2$.

Plug in:
$$T_{\text{classical}} = \frac{\sqrt 2\, \bar X}{S} = \frac{\sqrt 2 \cdot (X_1 + X_2)/2}{|X_1 - X_2|/\sqrt 2} = \frac{(X_1 + X_2)/\sqrt 2}{|X_1 - X_2|/\sqrt 2} = \frac{X_1 + X_2}{|X_1 - X_2|}.$$

Exactly $Z_1 / |Z_2|$.

**The classical $t$-statistic is the ratio of projection magnitudes in the rotated basis.** The one-sample $t$-test is the canonical linear model from §2, viewed in the original $(X_1, X_2)$ coordinate system instead of the rotated $(Z_1, Z_2)$ system.

This is the payoff of the geometric perspective: a template we can apply to many other testing problems just by choosing an appropriate orthonormal basis. We turn to that next.

## 4. One-Sample $t$-Test, General $n$

### 4.1 Setup and Notation

Let $X_1, \ldots, X_n \stackrel{iid}{\sim} N(\mu, \sigma^2)$. Test
$$H_0: \mu = 0 \quad \text{vs} \quad H_1: \mu \neq 0.$$

In vector form, $X \sim N_n(\mu \mathbf 1,\ \sigma^2 I_n)$ where $\mathbf 1 = (1, \ldots, 1)^T \in \mathbb R^n$ is the all-ones vector.

The mean vector $\mu \mathbf 1$ lies along the **signal direction** — the 1D subspace spanned by $\mathbf 1$. Let the unit vector in this direction be
$$\mathbf q_1 = \frac{\mathbf 1}{\sqrt n}.$$

Extend $\mathbf q_1$ to an orthonormal basis of $\mathbb R^n$ by choosing any $n \times (n-1)$ matrix $Q_r$ whose columns are orthonormal and orthogonal to $\mathbf q_1$. Then
$$Q = [\mathbf q_1\ |\ Q_r]$$
is an $n \times n$ orthogonal matrix.

**Notation.** Define
$$Z_1 = \mathbf q_1^T X \in \mathbb R, \qquad Z_r = Q_r^T X \in \mathbb R^{n-1}.$$

We'll call $Z_1$ the **signal coordinate** and $Z_r$ the **residual coordinates**. Together, $(Z_1, Z_r) = Q^T X$.

### 4.2 Rotated Coordinates

Compute $Z_1$ explicitly:
$$Z_1 = \mathbf q_1^T X = \frac{1}{\sqrt n} \sum_{i=1}^n X_i = \sqrt n\, \bar X.$$

By the affine transformation formula,
$$\binom{Z_1}{Z_r} = Q^T X \sim N_n\!\left(Q^T \mu \mathbf 1,\ \sigma^2 Q^T Q\right) = N_n\!\left(\binom{\sqrt n\, \mu}{\mathbf 0_{n-1}},\ \sigma^2 I_n\right),$$
where we used $Q^T \mathbf 1 = (\sqrt n,\ \mathbf 0_{n-1}^T)^T$ (since $\mathbf q_1^T \mathbf 1 = \sqrt n$ and the columns of $Q_r$ are orthogonal to $\mathbf 1$).

So:

- $Z_1 \sim N(\sqrt n\, \mu,\ \sigma^2)$.
- $Z_r \sim N_{n-1}(\mathbf 0,\ \sigma^2 I_{n-1})$ — regardless of $\mu$. The residual coordinates are pure noise.
- $Z_1 \perp\!\!\!\perp Z_r$.

### 4.3 Residual Magnitude

We want to use $\|Z_r\|^2$ as our variance estimate. Compute it using the orthogonal decomposition $\|X\|^2 = Z_1^2 + \|Z_r\|^2$:
$$\|Z_r\|^2 = \|X\|^2 - Z_1^2 = \sum_{i=1}^n X_i^2 - n \bar X^2.$$

Rearranging (via the standard identity),
$$\|Z_r\|^2 = \sum_{i=1}^n (X_i - \bar X)^2 = (n - 1) S^2,$$
where $S^2 = \tfrac{1}{n-1}\sum_i (X_i - \bar X)^2$ is the usual sample variance.

By rotational invariance applied to $Z_r \sim N_{n-1}(\mathbf 0, \sigma^2 I_{n-1})$,
$$\|Z_r\|^2 \sim \sigma^2 \chi^2_{n-1}.$$

Equivalently, $(n - 1) S^2 / \sigma^2 \sim \chi^2_{n-1}$. And because $Z_1 \perp\!\!\!\perp Z_r$, we automatically get $\bar X \perp\!\!\!\perp S^2$ — $\bar X$ depends only on $Z_1$, and $S^2$ depends only on $Z_r$.

### 4.4 Test Statistics

**$\sigma^2$ known.**
$$\frac{Z_1}{\sigma} = \frac{\sqrt n\, \bar X}{\sigma} \sim N(0, 1) \quad \text{under } H_0.$$
Reject for large $|Z_1/\sigma|$. This is the $z$-test.

**$\sigma^2$ unknown.** Standardize $Z_1$ using the residual variance estimate:
$$T = \frac{Z_1}{\sqrt{\|Z_r\|^2 / (n-1)}} = \frac{\sqrt n\, \bar X}{S} \sim t_{n-1} \quad \text{under } H_0.$$

This is exactly the classical one-sample $t$-statistic from Lecture 13 — now derived geometrically. Its squared form is
$$T^2 = \frac{Z_1^2}{\|Z_r\|^2 / (n-1)} \sim F_{1, n-1}.$$

### 4.5 Geometric Summary

The picture in $\mathbb R^n$:

- Decompose $\mathbb R^n$ into two orthogonal subspaces:
  - **Signal subspace** (1-dimensional): the line spanned by $\mathbf 1$.
  - **Residual subspace** ($(n-1)$-dimensional): the hyperplane orthogonal to $\mathbf 1$.
- $|Z_1|$ is the length of $X$'s projection onto the signal subspace.
- $\|Z_r\|$ is the length of $X$'s projection onto the residual subspace.
- The test statistic $T = Z_1 / \sqrt{\|Z_r\|^2/(n-1)}$ is a **ratio of projection magnitudes** (signal over normalized residual).

The division of $\|Z_r\|^2$ by $n - 1$ is exactly the adjustment needed to make the denominator an unbiased estimate of $\sigma^2$ — $E[\|Z_r\|^2/(n-1)] = \sigma^2$ follows from $E[\chi^2_{n-1}] = n - 1$.

This picture — **signal length over residual length** — is what we'll generalize on Thursday.

---

## 5. Preview: A Broader Framework ($n = 3$ Warm-Up)

### 5.1 Motivating Setup

Consider a new scenario: we observe three independent Gaussians,
$$Z_0, Z_1, Z_2 \text{ independent}, \qquad Z_j \sim N(\mu_j,\ \sigma^2),$$
with the following structure on the means:

- $E[Z_0] = \mu_0$ is a **nuisance parameter** — unknown, unrestricted under both $H_0$ and $H_1$.
- $E[Z_1] = \mu_1$ is what we want to test: $H_0: \mu_1 = 0$ vs $H_1: \mu_1 \neq 0$.
- $E[Z_2] = 0$ is pure noise under both hypotheses.

The new element here is $Z_0$: unlike §2, it has a free mean that we're not trying to test.

### 5.2 What Do We Do with $Z_0$?

Two natural questions.

**Can $Z_0$ help test $H_0: \mu_1 = 0$?** No. The mean of $Z_0$ is a free parameter under both hypotheses — $H_0$ and $H_1$ tell us nothing about $\mu_0$. And $Z_0 \perp\!\!\!\perp Z_1$, so seeing $Z_0$ gives us no information about $Z_1$ or $\mu_1$. Observing $Z_0$ doesn't help distinguish $H_0$ from $H_1$.

**Can $Z_0$ help estimate $\sigma^2$?** No. Unlike $Z_2$, whose mean is known to be $0$ under both hypotheses, the mean of $Z_0$ is *unknown*. So $Z_0^2$ is not an unbiased estimate of $\sigma^2$ — we'd need to subtract $\mu_0$ before squaring, and we don't know $\mu_0$.

**Conclusion.** $Z_0$ is useless. Drop it. The test is based on $(Z_1, Z_2)$ alone — exactly the machinery from §2.

This is the **key structural insight**: a nuisance direction with an unrestricted mean contributes nothing to the test statistic. It's neither signal (its mean isn't what we're testing) nor noise (its mean isn't known). We simply ignore it.

### 5.3 Preview of Thursday

On Thursday we'll generalize this picture to the **canonical linear model in full**: observe
$$Z \sim N_n(\mu,\ \sigma^2 I_n),$$
with the mean vector $\mu$ decomposed into three blocks of dimensions $d_0, d_1, d_r$ (with $d_0 + d_1 + d_r = n$):

- $Z_0 \in \mathbb R^{d_0}$: **nuisance** — $\mu_0$ unrestricted under both hypotheses.
- $Z_1 \in \mathbb R^{d_1}$: **tested** — $\mu_1 = 0$ under $H_0$, $\mu_1 \neq 0$ under $H_1$.
- $Z_r \in \mathbb R^{d_r}$: **pure noise** — mean $0$ under both hypotheses.

The tests split into four cases:

| | $\sigma^2$ known | $\sigma^2$ unknown |
|---|---|---|
| $d_1 = 1$ | $z$-test | $t_{d_r}$-test |
| $d_1 \geq 1$ | $\chi^2_{d_1}$-test | $F_{d_1, d_r}$-test |

Each is a ratio of projection magnitudes onto the signal and residual subspaces. All four are variations on the same geometric theme.

And — the payoff — a huge range of classical testing problems can be rotated into this form:

- **Two-sample $t$-test.** $X_1, \ldots, X_m \stackrel{iid}{\sim} N(\mu_A, \sigma^2)$ and $Y_1, \ldots, Y_n \stackrel{iid}{\sim} N(\mu_B, \sigma^2)$; test $\mu_A = \mu_B$.
- **One-way ANOVA.** Compare $k$ group means simultaneously.
- **Linear regression.** Test whether a subset of regression coefficients is zero.

All three turn out to be the canonical linear model with different choices of signal subspace. We'll see this Thursday.

---

## 6. Summary

| Concept | Key Idea |
|---|---|
| **$\chi^2, t, F$** | Built from iid standard normals: $\chi^2_d = \sum Z_i^2$; $t_d = Z/\sqrt{V/d}$; $F_{d_1, d_2} = (V_1/d_1)/(V_2/d_2)$; $t_d^2 \sim F_{1, d}$. |
| **Rotational invariance** | $N_n(0, \sigma^2 I_n)$ is unchanged by orthogonal rotations. Squared projection magnitudes onto orthogonal subspaces are independent $\sigma^2 \chi^2$. |
| **Canonical LM, $n=2$** | Test $\mu_1 = 0$ in $Z \sim N_2(\binom{\mu_1}{0}, \sigma^2 I_2)$: $z$/$\chi^2_1$ if $\sigma^2$ known, $t_1$/$F_{1,1}$ if not. |
| **Rotation** | The one-sample $t$-test is the canonical LM in disguise. Rotate by $Q = [\mathbf q_1\ |\ Q_r]$ to expose the structure. |
| **General $n$ one-sample $t$** | $T = \sqrt n\, \bar X / S = Z_1 / \sqrt{\|Z_r\|^2/(n-1)}$: ratio of projection magnitudes. $(n-1)S^2/\sigma^2 \sim \chi^2_{n-1}$ and $\bar X \perp\!\!\!\perp S^2$ follow automatically. |
| **Nuisance directions** | A direction whose mean is unrestricted under both $H_0$ and $H_1$ contributes nothing to the test — ignore it. |

**Key takeaways.**

1. Every test we did today is a **ratio of projection magnitudes** onto orthogonal subspaces. The $z$-, $\chi^2$-, $t$-, and $F$-tests are four cases of the same picture, distinguished by whether $\sigma^2$ is known and whether the signal subspace is 1D.

2. **Rotational invariance of $N_n(0, \sigma^2 I_n)$** is the engine: the affine transformation formula plus orthogonal $Q$ plus $\sigma^2 I_n$ covariance gives us independence of projections onto orthogonal subspaces and $\sigma^2 \chi^2$ marginals of squared projection magnitudes.

3. The classical one-sample $t$-test is a special case: rotate into the basis $[\mathbf q_1\ |\ Q_r]$, read off $Z_1 = \sqrt n \bar X$ and $\|Z_r\|^2 = (n-1) S^2$, form the ratio.

4. **Nuisance directions** — unrestricted means under both hypotheses — are ignored. Thursday we'll see how many familiar testing problems fit into this framework.

**Next time (Lecture 24):** the canonical linear model in full generality, with applications to the two-sample $t$-test, one-way ANOVA, and linear regression.